In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import re


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- col_ends_with ---
FIX_COL_ENDS_WITH_DATA_PD = pd.DataFrame({"policy_id": [1], "claim_id": [2], "claim_amt": [10.0]})
FIX_COL_ENDS_WITH_DATA_PL = pl.from_pandas(FIX_COL_ENDS_WITH_DATA_PD)
FIX_COL_ENDS_WITH_DATA = FIX_COL_ENDS_WITH_DATA_PD
FIX_COL_ENDS_WITH_PAT = "_id"
FIX_COL_ENDS_WITH_KWARGS = {}

# --- col_matches ---
FIX_COL_MATCHES_DATA_PD = pd.DataFrame({"Claim_A": [1], "claim_b": [2], "premium": [3]})
FIX_COL_MATCHES_DATA_PL = pl.from_pandas(FIX_COL_MATCHES_DATA_PD)
FIX_COL_MATCHES_DATA = FIX_COL_MATCHES_DATA_PD
FIX_COL_MATCHES_KWARGS = {"case": False, "regex": True}
FIX_COL_MATCHES_PAT = "claim"

# --- col_starts_with ---
FIX_COL_STARTS_WITH_DATA_PD = pd.DataFrame({"pol_num": [1], "pol_date": [2], "claim_id": [3]})
FIX_COL_STARTS_WITH_DATA_PL = pl.from_pandas(FIX_COL_STARTS_WITH_DATA_PD)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_col_ends_with(data, pat, kwargs):
    return list(data.columns[data.columns.str.endswith(pat, **kwargs)])
    return None

def before_col_matches(data, kwargs, pat):
    return list(data.columns[data.columns.str.contains(pat, **kwargs)])
    return None

def before_col_starts_with():
    def col_starts_with(data: pd.DataFrame,
                        pat: str,
                        **kwargs):
        return list(data.columns[data.columns.str.startswith(pat, **kwargs)])
    return col_starts_with

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_col_ends_with(data, pat, kwargs):
    return [col for col in data.columns if col.endswith(pat)]
    return None

def gen_col_matches(data, kwargs, pat):
    regex = kwargs.pop("regex", True)
    case = kwargs.pop("case", True)
    flags = kwargs.pop("flags", 0)

    if not case:
        flags |= re.IGNORECASE

    if regex:
        pattern = re.compile(pat, flags=flags)
        return [col for col in data.columns if pattern.search(col)]
    return [col for col in data.columns if re.search(re.escape(pat), col, flags=flags)]
    return flags

def gen_col_starts_with():
    def col_starts_with(data: pl.DataFrame,
                        pat: str,
                        **kwargs):
        na = kwargs.get("na", None)
        return [col for col in data.columns if col.startswith(pat)] if na is None else [col for col in data.columns if col.startswith(pat) or na]
    return col_starts_with

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: col_matches ===

try:
    _r = gen_col_matches(FIX_COL_MATCHES_DATA_PL, dict(FIX_COL_MATCHES_KWARGS), FIX_COL_MATCHES_PAT)
    print("✅ L1 smoke gen_col_matches: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_col_matches: {type(_e).__name__}: {_e}")

try:
    _rb = before_col_matches(FIX_COL_MATCHES_DATA_PD, dict(FIX_COL_MATCHES_KWARGS), FIX_COL_MATCHES_PAT)
    print("✅ L1 smoke before_col_matches: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_col_matches: {type(_e).__name__}: {_e}")

try:
    _rb = before_col_matches(FIX_COL_MATCHES_DATA_PD, dict(FIX_COL_MATCHES_KWARGS), FIX_COL_MATCHES_PAT)
    _rg = gen_col_matches(FIX_COL_MATCHES_DATA_PL, dict(FIX_COL_MATCHES_KWARGS), FIX_COL_MATCHES_PAT)
    print("✅ L2 equivalence col_matches: MATCH" if _rb == _rg else f"❌ L2 equivalence col_matches: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence col_matches: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = before_col_matches(FIX_COL_MATCHES_DATA_PD, {"regex": False, "case": True}, "claim")
    _rg = gen_col_matches(FIX_COL_MATCHES_DATA_PL, {"regex": False, "case": True}, "claim")
    print("✅ L3 edge col_matches literal case-sensitive: MATCH" if _rb == _rg else f"❌ L3 edge col_matches literal case-sensitive: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge col_matches: {type(_e).__name__}: {_e}")

try:
    _kwargs_before = {"regex": True, "case": True, "flags": re.IGNORECASE}
    _kwargs_gen = {"regex": True, "case": True, "flags": re.IGNORECASE}
    _rb = before_col_matches(FIX_COL_MATCHES_DATA_PD, _kwargs_before, "^claim")
    _rg = gen_col_matches(FIX_COL_MATCHES_DATA_PL, _kwargs_gen, "^claim")
    print("✅ L3 edge col_matches flags ignorecase: MATCH" if _rb == _rg else f"❌ L3 edge col_matches flags ignorecase: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge col_matches flags ignorecase: {type(_e).__name__}: {_e}")

try:
    _kwargs_before = {"regex": False, "case": False}
    _kwargs_gen = {"regex": False, "case": False}
    _rb = before_col_matches(FIX_COL_MATCHES_DATA_PD, _kwargs_before, "CLAIM")
    _rg = gen_col_matches(FIX_COL_MATCHES_DATA_PL, _kwargs_gen, "CLAIM")
    print("✅ L3 edge col_matches literal ignorecase: MATCH" if _rb == _rg else f"❌ L3 edge col_matches literal ignorecase: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge col_matches literal ignorecase: {type(_e).__name__}: {_e}")

try:
    _kwargs = {"regex": False, "case": False}
    _before_kwargs = dict(_kwargs)
    before_col_matches(FIX_COL_MATCHES_DATA_PD, _kwargs, "claim")
    _before_unchanged = _kwargs == _before_kwargs
    _kwargs = {"regex": False, "case": False}
    _gen_before = dict(_kwargs)
    gen_col_matches(FIX_COL_MATCHES_DATA_PL, _kwargs, "claim")
    _gen_unchanged = _kwargs == _gen_before
    if _before_unchanged == _gen_unchanged == True:
        print("✅ L3 edge col_matches kwargs not mutated: MATCH")
    else:
        print(f"❌ L3 edge col_matches kwargs not mutated: MISMATCH — before_unchanged={_before_unchanged}, gen_unchanged={_gen_unchanged}, gen_kwargs={_kwargs}")
except Exception as _e:
    print(f"❌ L3 edge col_matches kwargs mutation: {type(_e).__name__}: {_e}")
